# RAG01｜SysDoctor 技術知識庫建置

**作業特色**：自行撰寫 Zabbix、Linux、網路異常排查文件，使用老師指定的 SentenceTransformers、EmbeddingGemma 2、FAISS 建立向量資料庫。

資料性質：個人整理的教學指引，非官方操作手冊。

## 1. 安裝老師範例的固定套件

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"

## 2. 載入原始文件 ZIP
上傳附檔 `sysdoctor-docs.zip`。

In [ ]:
import json, zipfile, hashlib, tempfile, pathlib
import numpy as np
import faiss, torch
from sentence_transformers import SentenceTransformer
from IPython.display import display
from google.colab import files
MODEL_ID = "google/embeddinggemma-2"
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
print("裝置", DEVICE)
uploaded=files.upload()
assert "sysdoctor-docs.zip" in uploaded, "請上傳 sysdoctor-docs.zip"
with zipfile.ZipFile("sysdoctor-docs.zip") as z:
    paths=[p for p in z.namelist() if p.endswith(".txt") and "/" not in p]
    assert paths, "沒有 TXT 文件"
    documents=[]
    for idx,p in enumerate(sorted(paths),1):
        raw=z.read(p).decode("utf-8-sig")
        title=next((x.split("：",1)[1].strip() for x in raw.splitlines() if x.startswith("標題：")),p)
        text=raw.split("【內容】",1)[1].strip()
        documents.append({"doc_id":f"doc_{idx:03d}","text":f"{title}\n\n{text}","metadata":{"title":title,"filename":p,"document_type":"自行整理技術筆記","source_url":None}})
print("文件數",len(documents))
for d in documents: print(d['doc_id'],d['metadata']['title'])

## 3. 向量化文本並建立 FAISS 索引

In [ ]:
from huggingface_hub import HfApi
revision=HfApi().model_info(MODEL_ID,token=False).sha
model=SentenceTransformer(MODEL_ID,revision=revision,token=False,device=DEVICE,config_kwargs={"vision_config":None,"audio_config":None},model_kwargs={"dtype":DTYPE,"attn_implementation":"eager"})
inputs=[DOCUMENT_TEMPLATE.format(text=d['text']) for d in documents]
emb=model.encode(inputs,prompt="",batch_size=4,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
emb=np.ascontiguousarray(emb,dtype=np.float32)
faiss.normalize_L2(emb)
index=faiss.IndexFlatIP(emb.shape[1]);index.add(emb)
print("索引筆數",index.ntotal,"向量維度",index.d)

## 4. 檢索測試

In [ ]:
question="ZBX_TCP_READ() failed: [104] 是什麼問題？"
q=model.encode([QUERY_TEMPLATE.format(question=question)],prompt="",normalize_embeddings=True,convert_to_numpy=True)
q=np.ascontiguousarray(q,dtype=np.float32)
scores,ids=index.search(q,3)
for i,score in zip(ids[0],scores[0]):print(f"{documents[int(i)]['metadata']['title']}｜相似度 {score:.3f}")

## 5. 匯出 rag-data.zip
**請截圖**：索引筆數、檢索結果及下載檔案。

In [ ]:
out=pathlib.Path(tempfile.mkdtemp())
faiss.write_index(index,str(out/'index.faiss'))
(out/'documents.json').write_text(json.dumps(documents,ensure_ascii=False,indent=2),encoding='utf-8')
(out/'index_doc_ids.json').write_text(json.dumps([d['doc_id'] for d in documents],ensure_ascii=False),encoding='utf-8')
config={'format_version':'1.0','model_id':MODEL_ID,'model_revision':revision,'model_config_kwargs':{'vision_config':None,'audio_config':None},'dimension':int(index.d),'query_template':QUERY_TEMPLATE,'document_template':DOCUMENT_TEMPLATE,'encode_prompt':'','document_count':len(documents),'metric':'cosine_similarity_via_normalized_inner_product'}
(out/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
with zipfile.ZipFile('rag-data.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in out.iterdir(): z.write(p,arcname=p.name)
print('已產生 rag-data.zip')
files.download('rag-data.zip')

## AI 使用說明
使用生成式 AI 協助初版程式架構、提示詞及文字整理；本人需要理解、測試、修正，並在正式提交時附上 AI 對話 Prompt 與生成結果截圖。